# 11 toolchain

Question: does every job of `.github/workflows/ci.yml` that runs cargo install the toolchain of
`rust-toolchain.toml` with its components and its profile in one step before any step that runs
cargo, so that no cargo of the job makes rustup add a component on the fly?

A proxy of rustup, such as `cargo`, adds the components that `rust-toolchain.toml` names when the
toolchain lacks them, and two proxies that do so at once fail with
`detected conflict: 'bin/cargo-fmt'`. The notebooks of CI failed that way while the step installed
the channel alone. The notebook reads the steps of every job, then runs the step in a rustup home of
its own, which downloads the toolchain, and starts two cargo at once there, which must add nothing.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# rustup, a path relative to the root or a name on PATH. The notebook copies it into a home of its
# own and leaves the home of the user alone.
rustup = "rustup"
# The jobs of ci.yml that run cargo, and the name of the step that installs the toolchain.
cargo_jobs = ["rust", "dependencies", "notebooks"]
install_step = "Install the toolchain of rust-toolchain.toml"
# True runs the step in a fresh rustup home, which downloads the toolchain from static.rust-lang.org.
fresh_home = True

## Setup

The cell finds the repository, reads `ci.yml` and `rust-toolchain.toml` and prints the versions.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path

import yaml

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
WORKFLOW = yaml.safe_load((P.root / ".github" / "workflows" / "ci.yml").read_text())
TOOLCHAIN = tomllib.loads((P.root / "rust-toolchain.toml").read_text())["toolchain"]
RUSTUP = (P.root / rustup).resolve() if "/" in rustup else Path(shutil.which(rustup) or rustup)
assert RUSTUP.is_file(), "no rustup at %s" % rustup
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-11-"))

print("python:", sys.version.split()[0])
print("rustup:", run([RUSTUP, "--version"], check=True).stdout.strip())
print("toolchain:", TOOLCHAIN)
assert {"rustfmt", "clippy"} <= set(TOOLCHAIN.get("components", [])), "rust-toolchain.toml names no rustfmt and clippy"

## The steps of ci.yml

Every job that runs cargo, directly or through the notebooks, holds the step that installs the
toolchain, with one script in all of them, and the step comes before the first step that runs
cargo. The script names no channel, so rustup takes the channel, the components and the profile
from `rust-toolchain.toml`.

In [ ]:
SCRIPTS = set()
for name, job in WORKFLOW["jobs"].items():
    steps = [step.get("name") or step.get("uses") for step in job["steps"]]
    users = [index for index, step in enumerate(job["steps"]) if step.get("name") != install_step
             and any(word in (step.get("run") or "") for word in ("cargo", "tests/run.py"))]
    if name not in cargo_jobs:
        assert not users, "the job %s runs cargo but is none of %s" % (name, cargo_jobs)
        continue
    assert install_step in steps, "the job %s does not install the toolchain" % name
    installs = steps.index(install_step)
    print("%-13s installs the toolchain in step %d, runs cargo first in step %d" % (name, installs + 1, users[0] + 1))
    assert users and installs < users[0], "the job %s runs cargo before it installs the toolchain" % name
    SCRIPTS.add(job["steps"][installs]["run"])
assert len(SCRIPTS) == 1, "the jobs install the toolchain in different ways:\n%s" % "\n".join(SCRIPTS)
SCRIPT = SCRIPTS.pop()
print(SCRIPT)
assert SCRIPT.splitlines()[0].split() == ["rustup", "toolchain", "install"], "the step names a channel of its own"

## A rustup home of its own

The cell copies rustup into a home of its own, with its proxies, runs the script of the step there
with bash in a folder that holds a copy of `rust-toolchain.toml`, as the runner does, and asserts
that the toolchain holds rustfmt and clippy. Then it starts two cargo at once, which must both run
and download nothing.

In [ ]:
if fresh_home:
    cargo_home, rustup_home = SCRATCH / "cargo", SCRATCH / "rustup"
    (cargo_home / "bin").mkdir(parents=True)
    shutil.copy2(RUSTUP, cargo_home / "bin" / "rustup")
    for proxy in ("cargo", "rustc", "rustdoc", "rustfmt", "cargo-fmt", "cargo-clippy", "clippy-driver"):
        (cargo_home / "bin" / proxy).symlink_to("rustup")
    project = SCRATCH / "project"
    project.mkdir()
    shutil.copy2(P.root / "rust-toolchain.toml", project / "rust-toolchain.toml")
    environment = {key: value for key, value in os.environ.items() if key not in ("RUSTUP_TOOLCHAIN", "RUSTC", "CARGO")}
    environment.update(RUSTUP_HOME=str(rustup_home), CARGO_HOME=str(cargo_home),
                       PATH=os.pathsep.join([str(cargo_home / "bin"), os.environ["PATH"]]))

    def inside(argv, **options):
        return subprocess.run(argv, cwd=project, env=environment, capture_output=True, text=True, **options)

    assert inside([cargo_home / "bin" / "rustup", "set", "auto-self-update", "disable"]).returncode == 0
    step = inside(["bash", "--noprofile", "--norc", "-e", "-o", "pipefail", "-c", SCRIPT], timeout=1200)
    print(step.stdout.strip())
    assert step.returncode == 0, "the step failed:\n" + step.stdout + step.stderr
    installed = inside([cargo_home / "bin" / "rustup", "component", "list", "--installed"]).stdout.split()
    print("installed:", " ".join(sorted(installed)))
    size = sum(path.stat().st_size for path in rustup_home.rglob("*") if path.is_file() and not path.is_symlink())
    print("the rustup home holds %d MiB" % (size >> 20))
    for component in TOOLCHAIN["components"]:
        assert any(name.startswith(component + "-") for name in installed), "the step installed no %s" % component

    both = [subprocess.Popen(["cargo", "--version"], cwd=project, env=environment, stdout=subprocess.PIPE,
                             stderr=subprocess.PIPE, text=True) for _ in range(2)]
    results = [(process.wait(timeout=600), *process.communicate()) for process in both]
    for status, out, err in results:
        print("cargo at once: exit %d, %s" % (status, out.strip()))
        assert status == 0 and "downloading" not in err and "installing" not in err, err
else:
    print("fresh_home is False, the step ran in no home of its own")

## Verdict

The cell clears the temporary directory. A run that reaches it passed.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: every job of ci.yml that runs cargo installs the toolchain of rust-toolchain.toml with its components first")